In [ ]:
%pip install scanpy -q

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from anndata import AnnData
import scvelo as scv
import cellrank as cr
import scipy.sparse as sp
import petsc4py

In [ ]:
# Load and explore data
path = 'paper_code/Packed_Figure_Reproducibility/Source_Data_for_Figure/Figure6/SHARESEQ_brain_epitrace_obj_age_estimated_multiome.h5ad'
adata = sc.read_h5ad(path)

print(f"Data shape: {adata.shape}")
print(f"Obs columns: {adata.obs.columns.tolist()}")
print(f"Obsm: {list(adata.obsm.keys())}")

# Display metadata
print("\nMetadata preview:")
print(adata.obs.head())

# Check key columns
for col in ['EpiTraceAge_iterative', 'Cluster.Name']:
    if col in adata.obs.columns:
        print(f"\n{col}:")
        print(adata.obs[col].describe() if adata.obs[col].dtype != 'object' else adata.obs[col].value_counts())

# Look for trajectory-related columns
trajectory_cols = [col for col in adata.obs.columns if any(x in col.lower() for x in ['trace', 'cyto', 'stem'])]
if trajectory_cols:
    print(f"\nTrajectory-related cols: {trajectory_cols}")

# Check embeddings
if 'X_umap' in adata.obsm:
    print(f"UMAP shape: {adata.obsm['X_umap'].shape}")

In [ ]:
sc.settings.verbosity = 0
sc.settings.figdir = '.'

# Load and prepare data
path = 'paper_code/Packed_Figure_Reproducibility/Source_Data_for_Figure/Figure6/SHARESEQ_brain_epitrace_obj_age_estimated_multiome.h5ad'
adata = sc.read_h5ad(path)

# Filter to clusters 0-7
cluster_ids = range(8)
adata = adata[adata.obs['Cluster.Name'].isin(cluster_ids)].copy()

# Add cluster names
cluster_names = {0:'Radial Glia', 1:'Cyc. Prog', 2:'nIPC', 3:'ExN2', 4:'ExN3', 5:'ExN4', 6:'ExN5', 7:'mGPC/OPC'}
adata.obs['Real.Name'] = adata.obs['Cluster.Name'].map(cluster_names).astype('category')

# Set colors
colors = ["#FFA500", "#8B0000", "#98F5FF", "#8EE5EE", "#7AC5CD", "#53868B", "#A2CD5A", "#BCEE68"]
adata.uns['Real.Name_colors'] = colors

print(f"Ready: {adata.shape[0]} cells, {len(adata.obs['Real.Name'].cat.categories)} types\n")

# Plot Panel A
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.embedding(adata, basis='umap', color='Real.Name', ax=axs[0], 
                title='Panel A: Combined Trajectory', legend_loc='right', frameon=False, size=20, show=False)
sc.pl.embedding(adata, basis='umap', color='EpiTraceAge_iterative', ax=axs[1], 
                title='Panel A: EpiTrace Age', cmap='autumn_r', frameon=False, size=20, show=False)
plt.tight_layout()
plt.savefig('_panel_a.png', dpi=400, bbox_inches='tight')
plt.close()

# Plot Panel B
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

panels = [
    ('cytotrace_rna', 'B-i: CytoTRACE', 'viridis_r'),
    ('nFeature_peaks', 'B-ii: Peak Variability', 'plasma'),
    ('EpiTraceAge_iterative', 'B-iii: EpiTrace Age', 'autumn_r'),
]

for idx, (col, title, cmap) in enumerate(panels):
    sc.pl.embedding(adata, basis='umap', color=col, ax=axs[idx], 
                    title=title, cmap=cmap, frameon=False, size=20, show=False)

plt.tight_layout()
plt.savefig('_panel_b.png', dpi=400, bbox_inches='tight')
plt.close()

print("✅ Complete\nSaved: _panel_a.png, _panel_b.png")